In [44]:
import pandas as pd
import numpy as np

df = pd.read_csv("10th-toy-team3/final.csv.gz", compression="gzip")

print(df.shape)
df.head()

C:\Users\yooli\AppData\Local\Temp\ipykernel_3604\4264185814.py:4: DtypeWarning: Columns (17,22,34) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("10th-toy-team3/final.csv.gz", compression="gzip")


(56648, 156)


,age,ageCond,birthday,budam,buga1,buga2,buga3,chaksun1,chaksun2,chaksun3,...,style_vs_race,jkhr_starts,jkhr_winrate,tr_multi,wgBudam_chg,pl_harville,pl_disc,q_plc,gap_h,gap_d
0,2,2세,20210421,별정A,0,0,0,33000000,13200000,8400000,...,NaN,0,NaN,1.0,NaN,0.071700,0.094694,0.100378,0.028678,0.005684
1,2,2세,20210410,별정A,0,0,0,33000000,13200000,8400000,...,NaN,0,NaN,1.0,NaN,0.116673,0.139232,0.161834,0.045161,0.022602
2,2,2세,20210410,별정A,0,0,0,33000000,13200000,8400000,...,NaN,0,NaN,1.0,NaN,0.345707,0.327929,0.417362,0.071655,0.089433
3,2,2세,20210402,별정A,0,0,0,33000000,13200000,8400000,...,NaN,0,NaN,1.0,NaN,0.068898,0.091753,0.113284,0.044386,0.021531
4,2,2세,20210510,별정A,0,0,0,33000000,13200000,8400000,...,NaN,0,NaN,1.0,NaN,0.055747,0.077588,0.067777,0.012029,-0.009811


In [45]:
# =========================
# 1. 이변 점수 계산
# =========================

df["surprise_score"] = df["pop_pct"] - df["fin_pct"]

# =========================
# 2. 임계값 설정
#    절댓값 상위 5%
# =========================

threshold = np.quantile(
    np.abs(df["surprise_score"]),
    0.95
)

print("threshold =", threshold)

# =========================
# 3. 유형 A
#    다크호스
# =========================

df["darkhorse"] = (
    df["surprise_score"] >= threshold
).astype(int)

# =========================
# 4. 유형 B
#    인기마 붕괴
# =========================

df["favorite_bust"] = (
    df["surprise_score"] <= -threshold
).astype(int)

# =========================
# 5. 통합 이변 변수
# =========================

df["upset"] = (
    np.abs(df["surprise_score"]) >= threshold
).astype(int)

threshold = 0.6666666666666666


In [46]:
print("전체 이변 수")
print(df["upset"].sum())

print("다크호스 수")
print(df["darkhorse"].sum())

print("인기마 붕괴 수")
print(df["favorite_bust"].sum())

전체 이변 수
2985
다크호스 수
1282
인기마 붕괴 수
1703


In [47]:
# 다크호스 TOP 20
darkhorse_cases = (
    df.sort_values(
        "surprise_score",
        ascending=False
    )
    .head(20)
)

print("다크호스 TOP 20")
darkhorse_cases[
    [
        "hrName",
        "pop_pct",
        "fin_pct",
        "surprise_score",
        "ord",
        "pop_rank"
    ]
]

다크호스 TOP 20


,hrName,pop_pct,fin_pct,surprise_score,ord,pop_rank
43513,새내파워풀,1.0,0.0,1.0,1,11.0
11627,원더풀앤도버,1.0,0.0,1.0,1,12.0
37691,스마트스마일,1.0,0.0,1.0,1,11.0
31184,런던에이스,1.0,0.0,1.0,1,10.0
30900,푸에르테,1.0,0.0,1.0,1,12.0
49217,케이엔라이트,1.0,0.0,1.0,1,11.0
55182,블루레인,1.0,0.0,1.0,0,11.0
47386,환희뱅크,1.0,0.0,1.0,1,12.0
54836,맥스더드래건,1.0,0.0,1.0,1,11.0
37719,스타티세,1.0,0.0,1.0,1,8.0


In [48]:
# 인기마 붕괴 TOP 20
favorite_bust_cases = (
    df.sort_values(
        "surprise_score",
        ascending=True
    )
    .head(20)
)

print("인기마 붕괴 TOP 20")
favorite_bust_cases[
    [
        "hrName",
        "pop_pct",
        "fin_pct",
        "surprise_score",
        "ord",
        "pop_rank"
    ]
]

인기마 붕괴 TOP 20


,hrName,pop_pct,fin_pct,surprise_score,ord,pop_rank
20129,베스트조이,0.0,1.0,-1.0,10,1.0
17839,글로리강서,0.0,1.0,-1.0,10,1.0
40179,레이디오브윈,0.0,1.0,-1.0,11,1.0
45784,환상의나라,0.0,1.0,-1.0,11,1.0
16437,부산투데이,0.0,1.0,-1.0,9,1.0
43600,용비패왕,0.0,1.0,-1.0,12,1.0
22235,시대의명작,0.0,1.0,-1.0,10,1.0
14147,더플레이어,0.0,1.0,-1.0,8,1.0
21168,도끼불패,0.0,1.0,-1.0,10,1.0
45890,프라임크라운,0.0,1.0,-1.0,9,1.0


In [49]:
# 다크호스 특징 찾기

In [50]:
analysis_df = df.copy()

In [51]:
# 데이터 정리
result_cols = [
    "ord",
    "fin_rank",
    "fin_pct",
    "win",
    "place",
    "resid"
]

label_cols = [
    "darkhorse",
    "upset",
    "upset_A",
    "upset_B",
    "favorite_bust",
    "surprise_score"
]

market_cols = [
    "winOdds",
    "plcOdds",
    "p_raw",
    "q",
    "q_plc",
    "log_q",
    "logit_q",
    "pop_rank",
    "pop_pct",
    "pl_harville",
    "pl_disc",
    "gap_h",
    "gap_d"
]

id_cols = [
    "race_id",
    "hrNo",
    "hrName",
    "jkNo",
    "jkName",
    "trNo",
    "trName",
    "owNo",
    "owName"
]

drop_cols = (
    result_cols
    + label_cols
    + market_cols
    + id_cols
)

drop_cols = [
    c for c in drop_cols
    if c in analysis_df.columns
]

analysis_df = analysis_df.drop(
    columns=drop_cols
)

In [52]:
# 다크호스 vs 일반마 비교용 데이터
dark = analysis_df[df["darkhorse"] == 1]
normal = analysis_df[df["darkhorse"] == 0]

print(dark.shape)
print(normal.shape)

(1282, 125)
(55366, 125)


In [53]:
# Cohen's d 계산
num_cols = analysis_df.select_dtypes(
    include=np.number
).columns

effect_result = []

for col in num_cols:

    try:

        d = cohens_d(
            dark[col],
            normal[col]
        )

        effect_result.append(
            [col, d, abs(d)]
        )

    except:
        pass

effect_df = pd.DataFrame(
    effect_result,
    columns=[
        "feature",
        "cohens_d",
        "abs_d"
    ]
)

effect_df.sort_values(
    "abs_d",
    ascending=False
).head(15)

,feature,cohens_d,abs_d
44,hr_last_poppct,0.629637,0.629637
43,hr_last_finpct,0.629561,0.629561
42,hr_last_ord,0.591087,0.591087
77,age__z,0.489967,0.489967
85,jk_winrate__z,-0.481130,0.481130
78,age__pr,0.479896,0.479896
86,jk_winrate__pr,-0.471872,0.471872
79,rating__z,-0.439572,0.439572
81,hr_winrate__z,-0.439010,0.439010
80,rating__pr,-0.435304,0.435304


In [54]:
important_cols = [
    "hr_last_ord",
    "hr_last_finpct",
    "hr_last_poppct",
    "age",
    "jk_winrate",
    "jk_plcrate",
    "rating__pr"
]

summary = []

for col in important_cols:

    summary.append([
        col,
        dark[col].mean(),
        normal[col].mean()
    ])

summary_df = pd.DataFrame(
    summary,
    columns=[
        "feature",
        "dark_mean",
        "normal_mean"
    ]
)

summary_df

,feature,dark_mean,normal_mean
0,hr_last_ord,7.480607,5.647563
1,hr_last_finpct,0.674757,0.478628
2,hr_last_poppct,0.682248,0.483864
3,age,4.280811,3.760557
4,jk_winrate,0.073187,0.096382
5,jk_plcrate,0.238913,0.286753
6,rating__pr,0.367214,0.502552


In [55]:
# 인기마 붕괴 특징 찾기

In [56]:
analysis_df = df.copy()

drop_cols = [
    # 결과
    "ord",
    "fin_rank",
    "fin_pct",
    "win",
    "place",
    "resid",

    # 라벨
    "darkhorse",
    "favorite_bust",
    "upset",
    "upset_A",
    "upset_B",
    "surprise_score",

    # 시장
    "winOdds",
    "plcOdds",
    "p_raw",
    "q",
    "q_plc",
    "log_q",
    "logit_q",
    "pop_rank",
    "pop_pct",
    "pl_harville",
    "pl_disc",
    "gap_h",
    "gap_d",

    # 식별자
    "hrName"
]

drop_cols = [c for c in drop_cols if c in analysis_df.columns]

analysis_df = analysis_df.drop(columns=drop_cols)

In [57]:
bust_df = df[df["favorite_bust"] == 1].copy()
non_bust_df = df[df["favorite_bust"] == 0].copy()

print("인기마 붕괴 :", len(bust_df))
print("기타 :", len(non_bust_df))

인기마 붕괴 : 1703
기타 : 54945


In [58]:
# Cohen's d 계산
bust_num_cols = analysis_df.select_dtypes(
    include=np.number
).columns

bust_effect_result = []

for col in bust_num_cols:

    try:

        d = cohens_d(
            bust_df[col],
            non_bust_df[col]
        )

        bust_effect_result.append(
            [col, d, abs(d)]
        )

    except:
        pass

In [59]:
bust_effect_df = pd.DataFrame(
    bust_effect_result,
    columns=[
        "feature",
        "cohens_d",
        "abs_d"
    ]
)

bust_effect_df = (
    bust_effect_df
    .sort_values(
        "abs_d",
        ascending=False
    )
)

bust_effect_df.head(20)

,feature,cohens_d,abs_d
25,is_fav,0.704617,0.704617
45,hr_last_poppct,-0.687272,0.687272
82,hr_winrate__z,0.687197,0.687197
28,hr_plcrate,0.611776,0.611776
83,hr_winrate__pr,0.606530,0.606530
44,hr_last_finpct,-0.590365,0.590365
86,jk_winrate__z,0.569223,0.569223
43,hr_last_ord,-0.567871,0.567871
27,hr_winrate,0.552545,0.552545
32,jk_plcrate,0.545970,0.545970


In [60]:
bust_top_features = (
    bust_effect_df
    .head(10)["feature"]
    .tolist()
)

bust_summary = []

for col in bust_top_features:

    bust_summary.append([
        col,
        bust_df[col].mean(),
        non_bust_df[col].mean()
    ])

bust_summary_df = pd.DataFrame(
    bust_summary,
    columns=[
        "feature",
        "bust_mean",
        "normal_mean"
    ]
)

bust_summary_df

,feature,bust_mean,normal_mean
0,is_fav,0.296536,0.090126
1,hr_last_poppct,0.279030,0.495039
2,hr_winrate__z,0.626851,-0.021644
3,hr_plcrate,0.471760,0.306555
4,hr_winrate__pr,0.632476,0.473798
5,hr_last_finpct,0.304973,0.488769
6,jk_winrate__z,0.522821,-0.016226
7,hr_last_ord,3.983995,5.743620
8,hr_winrate,0.196457,0.104680
9,jk_plcrate,0.344257,0.283851


In [62]:
effect_df.head(15)          # 다크호스
bust_effect_df.head(15)     # 인기마 붕괴

,feature,cohens_d,abs_d
25,is_fav,0.704617,0.704617
45,hr_last_poppct,-0.687272,0.687272
82,hr_winrate__z,0.687197,0.687197
28,hr_plcrate,0.611776,0.611776
83,hr_winrate__pr,0.606530,0.606530
44,hr_last_finpct,-0.590365,0.590365
86,jk_winrate__z,0.569223,0.569223
43,hr_last_ord,-0.567871,0.567871
27,hr_winrate,0.552545,0.552545
32,jk_plcrate,0.545970,0.545970
